In [2]:
# Code to map Maven Hepatitis elements to Origami ALT Liver Enzyme elements
# Using semantic similarity library
# Author: Sandeep Chintabathina
# April 2026

In [3]:
# For each Maven Hepatitis data element, check if there is an exact or close matching data element in the origami ALT Liver Enzyme package
# The output will list the maven element alongside the origami element and other supporting attributes

In [4]:
from sentence_transformers import SentenceTransformer


In [5]:
import pandas as pd
import numpy as np

In [6]:
# Read maven elements
df_maven = pd.read_csv('output/maven_all_with_mmg_and_cste.csv',na_filter=False)
len(df_maven)

9501

In [7]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9501 entries, 0 to 9500
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Model                  9501 non-null   object
 1   question_number        9501 non-null   int64 
 2   depth                  9501 non-null   int64 
 3   Question               9501 non-null   object
 4   Question.Package.ID    9501 non-null   object
 5   Question.Package.Name  9501 non-null   object
 6   Message                9501 non-null   object
 7   mmg_element            9501 non-null   object
 8   cste_element           9501 non-null   object
dtypes: int64(2), object(7)
memory usage: 668.2+ KB


In [8]:
def clean_up(name):
    new_name =''
    for ch in name:
        # If space, digits, uppercase letters, lowercase letters, or underscore, add it to new name otherwise ignore
        if (ord(ch)==32 or 48<=ord(ch)<=57 or 65<=ord(ch)<=90 or 97<=ord(ch)<=122 or ord(ch)==95 ):
            new_name+=ch
    return new_name.strip()

In [9]:
clean_up('[sdfs] 34_AS ^#~')

'sdfs 34_AS'

In [10]:
# Clean up questions or data elements
df_maven.loc[:,'Question'] =[ clean_up(q) for q in df_maven.loc[:,'Question']]

In [11]:
df_maven.loc[:,'Model'].value_counts(dropna=False)

Model
General                  3866
STD                      2425
Coinfection               923
Hepatitis                 901
Childhood Lead            606
Cluster                   425
Lead investigation        255
STD EHR and screening      85
Aggregate                  15
Name: count, dtype: int64

In [12]:
df_maven_hep = df_maven[df_maven['Model']=='Hepatitis']
len(df_maven_hep)

901

In [13]:
# No duplicates
df_maven_hep[df_maven_hep.duplicated(['Question'],keep=False)]

,Model,question_number,depth,Question,Question.Package.ID,Question.Package.Name,Message,mmg_element,cste_element


In [14]:
# treatment elements
df_alt = pd.read_csv('origami/csv/Custom__ALTLiverEnzyme.csv',na_filter=False)
len(df_alt)

48

In [15]:
df_alt.loc[:,'Label'].value_counts(dropna=False)

Label
                                                  37
ALT Liver Enzyme ID                                1
Client                                             1
Entry User                                         1
Entry Date                                         1
Modified User                                      1
Modified Date                                      1
[INV826_alt]Date of ALT Res                        1
[1742-6]ALT [SGPT] Result                          1
[INV827_alt]ALT Test Result Upper Limit Normal     1
Incident Type                                      1
Incident                                           1
Name: count, dtype: int64

In [16]:
# Get the two columns of interest and non-blank values
df_alt = df_alt[['Field','Label']]
df_alt = df_alt[df_alt['Label']!='']
len(df_alt)

11

In [17]:
df_alt = df_alt.reset_index(drop=True)

In [18]:
df_alt

,Field,Label
0,ALTLiverEnzymeID,ALT Liver Enzyme ID
1,ClientID,Client
2,EntryUserID,Entry User
3,EntryDate,Entry Date
4,ModifiedUserID,Modified User
5,ModifiedDate,Modified Date
6,CustomDate1,[INV826_alt]Date of ALT Res
7,CustomNumber1,[1742-6]ALT [SGPT] Result
8,CustomNumber2,[INV827_alt]ALT Test Result Upper Limit Normal
9,IncidentTypeID,Incident Type


In [19]:
# Original label to be used later for output
df_alt.rename(columns={'Label':'original_label'},inplace=True)
df_alt.columns

Index(['Field', 'original_label'], dtype='object')

In [ ]:
# Remove only the first occurrence of [] in a string
def remove_squares(somestring):
    if ']' in somestring:
        for i in range(len(somestring)):
            if somestring[i]==']':
                return somestring[i+1:].strip()
    return somestring.strip()


In [28]:
remove_squares('asd[asdasd]asc[obas]da')

'asc[obas]da'

In [29]:
# Process the column values (eliminate the values in square brackets)
df_alt.loc[:,'Label'] = [ remove_squares(label) for label in df_alt.loc[:,'original_label']]

In [30]:
# Clean up column names in both data sets
df_alt.loc[:,'Label'] =[ clean_up(q) for q in df_alt.loc[:,'Label']]

In [31]:
df_alt.loc[:,'Label'].value_counts()

Label
ALT Liver Enzyme ID                   1
Client                                1
Entry User                            1
Entry Date                            1
Modified User                         1
Modified Date                         1
Date of ALT Res                       1
ALT SGPT Result                       1
ALT Test Result Upper Limit Normal    1
Incident Type                         1
Incident                              1
Name: count, dtype: int64

In [ ]:
# duplicates labels do not exist 
df_alt[df_alt.duplicated(['Label'],keep=False)]

,Field,original_label,Label


In [ ]:
# dropping them
#df_alt = df_alt.drop_duplicates(['Label'])

In [ ]:
#df_alt[df_alt.duplicated(['Label'],keep=False)]

,Field,original_label,Label


In [33]:
df_alt = df_alt.reset_index(drop=True)
df_alt

,Field,original_label,Label
0,ALTLiverEnzymeID,ALT Liver Enzyme ID,ALT Liver Enzyme ID
1,ClientID,Client,Client
2,EntryUserID,Entry User,Entry User
3,EntryDate,Entry Date,Entry Date
4,ModifiedUserID,Modified User,Modified User
5,ModifiedDate,Modified Date,Modified Date
6,CustomDate1,[INV826_alt]Date of ALT Res,Date of ALT Res
7,CustomNumber1,[1742-6]ALT [SGPT] Result,ALT SGPT Result
8,CustomNumber2,[INV827_alt]ALT Test Result Upper Limit Normal,ALT Test Result Upper Limit Normal
9,IncidentTypeID,Incident Type,Incident Type


In [34]:
# Do a semantic match between Maven Hep elements and Origami Treatment labels

In [35]:
maven_elements = [str.replace(maven_de,'_',' ') for maven_de in df_maven_hep.Question]
origami_elements = [str.replace(col,'_',' ') for col in df_alt.Label]

In [493]:
#origami_elements

In [36]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
# Compute embeddings
embeddings_maven = model.encode(maven_elements)
embeddings_origami = model.encode(origami_elements)

In [38]:
print(len(embeddings_origami))
print(len(embeddings_maven))
embeddings_origami
embeddings_maven

11
901


array([[-0.04004316,  0.03481774, -0.02303074, ...,  0.06413301,
         0.07289726,  0.05008497],
       [ 0.01921209,  0.03713002,  0.04779667, ...,  0.04271896,
        -0.05558128, -0.02244971],
       [-0.1053156 ,  0.01508899,  0.01699461, ...,  0.08866485,
        -0.01472143, -0.02046957],
       ...,
       [-0.08527962,  0.04394535, -0.00333766, ..., -0.00748747,
         0.13976483, -0.05983388],
       [-0.10253949,  0.03828218, -0.01935591, ..., -0.01946716,
         0.04943707,  0.04164878],
       [-0.04428944,  0.05000523, -0.03245968, ..., -0.02810097,
        -0.05133276,  0.00550872]], dtype=float32)

In [39]:
# Compute cosine similarities
similarities = model.similarity(embeddings_maven,embeddings_origami)

In [40]:
similarities

tensor([[ 0.1979,  0.2485,  0.1516,  ...,  0.0578,  0.2570,  0.3304],
        [ 0.0330,  0.1044,  0.1415,  ..., -0.0518,  0.2337,  0.2941],
        [ 0.0625,  0.0654,  0.0365,  ...,  0.0234,  0.1409,  0.2257],
        ...,
        [ 0.2938,  0.1939,  0.0975,  ...,  0.1291,  0.1293,  0.1324],
        [ 0.1297,  0.0415,  0.1275,  ...,  0.0049,  0.0348, -0.0538],
        [ 0.1420,  0.0944,  0.0472,  ...,  0.0347,  0.2411,  0.1708]])

In [41]:
print(type(similarities))
len(similarities)

<class 'torch.Tensor'>


901

In [42]:
df_similarities = pd.DataFrame(similarities,index=df_maven_hep.Question,columns=df_alt.original_label)

In [43]:
df_similarities.to_csv('output/similarity_scores_ALTLiver.csv')

In [502]:
#df_similarities.loc['DEVELOPMENTAL_SCREENING',:]

In [44]:
df_maven_hep = df_maven_hep.reset_index(drop=True)
df_maven_hep.columns

Index(['Model', 'question_number', 'depth', 'Question', 'Question.Package.ID',
       'Question.Package.Name', 'Message', 'mmg_element', 'cste_element'],
      dtype='object')

In [45]:
df_output = df_maven_hep[['Model','depth','Question','Message','mmg_element','cste_element']]

In [40]:
#df_similarities.info()

In [46]:
#For every maven element (index), identify origami elements (column) that is the best match

filtered=[]
for idx in df_similarities.index:
    # Looking for columns with score higher than 0.92
    v = df_similarities.loc[idx,:]>0.75

    # Get the column with the maximum score
    max=0.0
    max_col=''
    for i,col in enumerate(df_similarities.columns):
        if v.iloc[i]==True:
            #print('value',df_similarities.loc[idx,col])
            if (df_similarities.loc[idx,col] > max):
                max = df_similarities.loc[idx,col]
                max_col=col

    #filtered.append({col:df_similarities.loc[idx,col]})
    #print(idx+','+max_col+':'+str(max)+'\n')
    filtered.append({'Question':idx,'target_label':max_col})



In [47]:
df_filtered = pd.DataFrame(filtered)
print(len(df_filtered))
#df_filtered.loc[:,'target_label'].value_counts(dropna=False)

901


In [48]:
df_merge = pd.merge(df_filtered,df_alt,left_on='target_label',right_on='original_label',how='left')

In [49]:
df_merge

,Question,target_label,Field,original_label,Label
0,DISEASE,,NaN,NaN,NaN
1,EVENT_DATE,,NaN,NaN,NaN
2,EPISODE_DATE,,NaN,NaN,NaN
3,EPISODE_DATE_BASED_ON,,NaN,NaN,NaN
4,INVESTIGATION_STATUS,,NaN,NaN,NaN
...,...,...,...,...,...
896,LEGACY_CONTACT,,NaN,NaN,NaN
897,LEGACY_PHC,,NaN,NaN,NaN
898,LEGACY_LAB,,NaN,NaN,NaN
899,LEGACY_FORM_DATA,,NaN,NaN,NaN


In [50]:
# merge the above merged df with output dataframe
df_final = pd.merge(df_output,df_merge,on='Question')
#df_final

In [51]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 901 entries, 0 to 900
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Model           901 non-null    object
 1   depth           901 non-null    int64 
 2   Question        901 non-null    object
 3   Message         901 non-null    object
 4   mmg_element     901 non-null    object
 5   cste_element    901 non-null    object
 6   target_label    901 non-null    object
 7   Field           2 non-null      object
 8   original_label  2 non-null      object
 9   Label           2 non-null      object
dtypes: int64(1), object(9)
memory usage: 70.5+ KB


In [52]:
# Add, rename, remove columns as needed
df_final['target_table'] = ['' if f=='' else 'Custom__ALTLiverEnzyme' for f in df_final.target_label]
df_final=df_final.rename(columns={'Field':'target_field'})
df_final=df_final.drop('Label',axis=1)


In [53]:
df_final = df_final[["Model",'Question','target_table','target_field','target_label','mmg_element','cste_element','Message','depth']]
#df_final

In [54]:
df_final

,Model,Question,target_table,target_field,target_label,mmg_element,cste_element,Message,depth
0,Hepatitis,DISEASE,,NaN,,,,,0
1,Hepatitis,EVENT_DATE,,NaN,,,,,0
2,Hepatitis,EPISODE_DATE,,NaN,,,,Episode date,0
3,Hepatitis,EPISODE_DATE_BASED_ON,,NaN,,,,Episode date based on,0
4,Hepatitis,INVESTIGATION_STATUS,,NaN,,,,,0
...,...,...,...,...,...,...,...,...,...
896,Hepatitis,LEGACY_CONTACT,,NaN,,,,,0
897,Hepatitis,LEGACY_PHC,,NaN,,,,,0
898,Hepatitis,LEGACY_LAB,,NaN,,,,,0
899,Hepatitis,LEGACY_FORM_DATA,,NaN,,,,,0


In [516]:
#df_final[df_final['target_field'].isna()]

In [55]:
df_final.to_csv('output/maven_ALTLiver_origami_similarity_match.csv',index=False)